In [1]:
import numpy as np
import pandas as pd
import ast
from shapely.geometry import Polygon

In [2]:
rooms_df = pd.read_csv("rooms_dataset.csv")

rooms_df.head()

,room_type,area,centroid_x,centroid_y,num_vertices,points,plan_id
0,Outdoor Terrace,248496.0296,1260.503333,387.083333,6,"[(1030.81, 507.29), (1030.81, 146.67), (1719.8...",6044
1,Entry Lobby,54825.5028,1414.466667,1211.093333,9,"[(1469.34, 1137.48), (1469.34, 1148.48), (1530...",6044
2,Kitchen,67064.8976,1576.103333,1320.606667,6,"[(1712.49, 1478.05), (1474.05, 1478.05), (1474...",6044
3,LivingRoom,221476.7801,1482.916667,934.763333,6,"[(1712.49, 1137.48), (1469.34, 1137.48), (1334...",6044
4,Utility Laundry,20826.7515,1121.873333,1377.933333,6,"[(1190.03, 1478.05), (1051.17, 1478.05), (1051...",6044


In [3]:
PIXEL_TO_METER = 0.01
AREA_SCALE = PIXEL_TO_METER ** 2

rooms_df["points"] = rooms_df["points"].apply(ast.literal_eval)

rooms_df["polygon"] = rooms_df["points"].apply(lambda pts: Polygon(pts).buffer(0))

# convert pixel area → m²
rooms_df["area"] = rooms_df["area"] * AREA_SCALE

In [4]:
def compute_aspect_ratio(polygon):

    minx, miny, maxx, maxy = polygon.bounds

    width = maxx - minx
    height = maxy - miny

    if min(width, height) == 0:
        return 0

    return max(width, height) / min(width, height)

In [5]:
def compute_compactness(polygon):

    area = polygon.area
    perimeter = polygon.length

    if perimeter == 0:
        return 0

    return (4 * np.pi * area) / (perimeter ** 2)

In [6]:
def compute_rectangularity(polygon):

    area = polygon.area

    minx, miny, maxx, maxy = polygon.bounds

    bbox_area = (maxx - minx) * (maxy - miny)

    if bbox_area == 0:
        return 0

    return area / bbox_area

In [7]:
rooms_df["aspect_ratio"] = rooms_df["polygon"].apply(compute_aspect_ratio)

rooms_df["compactness"] = rooms_df["polygon"].apply(compute_compactness)

rooms_df["rectangularity"] = rooms_df["polygon"].apply(compute_rectangularity)

In [8]:
plan_features = []

for plan_id, group in rooms_df.groupby("plan_id"):

    features = {}

    features["plan_id"] = plan_id

    features["num_rooms"] = len(group)

    features["avg_room_area"] = group["area"].mean()

    features["std_room_area"] = group["area"].std()

    features["avg_aspect_ratio"] = group["aspect_ratio"].mean()

    features["avg_compactness"] = group["compactness"].mean()

    features["avg_rectangularity"] = group["rectangularity"].mean()

    plan_features.append(features)

In [9]:
geometry_features_df = pd.DataFrame(plan_features)

geometry_features_df.head()

,plan_id,num_rooms,avg_room_area,std_room_area,avg_aspect_ratio,avg_compactness,avg_rectangularity
0,1,14,7.487408,6.326511,1.789442,0.649941,0.975016
1,2,7,6.913880,6.906330,2.267390,0.669636,0.997796
2,3,5,8.834605,9.940902,1.923613,0.673939,0.960626
3,4,5,12.932369,9.314597,1.916750,0.630721,0.937192
4,6,7,9.376730,5.578743,1.693088,0.689853,0.947068


In [10]:
geometry_features_df.to_csv("geometry_features.csv", index=False)